In [0]:

import pyspark.sql.functions as F
from datetime import datetime
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("STOCK MARKET PIPELINE - MONITORING & DASHBOARDS")
print("=" * 70)

# Cell 1: Load Gold Tables


In [0]:

gold_signals = spark.table(f"{catalog}.{schema}.gold_trading_signals")
gold_perf = spark.table(f"{catalog}.{schema}.gold_stock_performance")
gold_volatility = spark.table(f"{catalog}.{schema}.gold_volatility_analysis")
gold_momentum = spark.table(f"{catalog}.{schema}.gold_momentum_analysis")
 
print("✅ Loaded all gold tables for monitoring")
 
 

# Cell 2: Business Metrics


In [0]:

print("\n" + "=" * 70)
print("BUSINESS METRICS")
print("=" * 70 + "\n")
 
# Metric 1: Strong Buy Signals
strong_buys = gold_signals.filter(F.col("Trading_Signal") == "STRONG_BUY").count()
print(f"📈 Strong BUY signals: {strong_buys}")
 
# Metric 2: Strong Sell Signals
strong_sells = gold_signals.filter(F.col("Trading_Signal") == "STRONG_SELL").count()
print(f"📉 Strong SELL signals: {strong_sells}")
 
# Metric 3: Overbought stocks (RSI > 70)
overbought = gold_momentum.filter(F.col("RSI_Status") == "Overbought").count()
print(f"⚠️ Overbought stocks (RSI>70): {overbought}")
 
# Metric 4: Oversold stocks (RSI < 30)
oversold = gold_momentum.filter(F.col("RSI_Status") == "Oversold").count()
print(f"⚠️ Oversold stocks (RSI<30): {oversold}")
 
# Metric 5: Top performers
top_performers = gold_perf.filter(F.col("Performance_Tier") == "Top 3").count()
print(f"🏆 Top 3 performers: {top_performers}")
 
# Metric 6: Underperformers
underperformers = gold_perf.filter(F.col("Performance_Tier") == "Underperformer").count()
print(f"📉 Underperformers: {underperformers}")
 
# Metric 7: High volatility
high_vol = gold_volatility.filter(F.col("Volatility_Status") == "High Volatility").count()
print(f"📊 High volatility stocks: {high_vol}")
 
# Metric 8: Average confidence score
avg_confidence = gold_signals.agg(F.avg("Confidence")).collect()[0][0]
print(f"🎯 Average signal confidence: {round(avg_confidence, 2)}%")
 

# Cell 3: Market Sentiment


In [0]:

print("\n" + "=" * 70)
print("MARKET SENTIMENT")
print("=" * 70 + "\n")
 
signal_distribution = gold_signals.groupBy("Trading_Signal").count().orderBy(F.desc("count"))
 
print("Trading Signal Distribution:")
display(signal_distribution)
 
# Calculate market sentiment
bullish = gold_signals.filter(
    F.col("Trading_Signal").isin(["STRONG_BUY", "BUY", "ACCUMULATE"])
).count()
 
bearish = gold_signals.filter(
    F.col("Trading_Signal").isin(["STRONG_SELL", "SELL", "DISTRIBUTE"])
).count()
 
neutral = gold_signals.filter(F.col("Trading_Signal") == "HOLD").count()
 
total_signals = gold_signals.count()
bullish_pct = round(bullish / total_signals * 100, 2) if total_signals > 0 else 0
bearish_pct = round(bearish / total_signals * 100, 2) if total_signals > 0 else 0
neutral_pct = round(neutral / total_signals * 100, 2) if total_signals > 0 else 0
 
print(f"\n📊 OVERALL MARKET SENTIMENT:")
print(f"   Bullish: {bullish_pct}% ({bullish} stocks)")
print(f"   Bearish: {bearish_pct}% ({bearish} stocks)")
print(f"   Neutral: {neutral_pct}% ({neutral} stocks)")
 
if bullish_pct > bearish_pct + 10:
    print(f"\n🚀 Market Sentiment: STRONGLY BULLISH")
elif bullish_pct > bearish_pct:
    print(f"\n📈 Market Sentiment: BULLISH")
elif bearish_pct > bullish_pct + 10:
    print(f"\n🔴 Market Sentiment: STRONGLY BEARISH")
elif bearish_pct > bullish_pct:
    print(f"\n📉 Market Sentiment: BEARISH")
else:
    print(f"\n➡️ Market Sentiment: NEUTRAL/MIXED")
 

# Cell 4: Performance Rankings

In [0]:

print("\n" + "=" * 70)
print("PERFORMANCE RANKINGS (1-Month Return)")
print("=" * 70 + "\n")
 
top_5 = gold_perf.orderBy(F.desc("Return_1M_Percent")).limit(5).select(
    "Ticker", "Return_1M_Percent", "Performance_Tier"
)
 
print("✅ TOP 5 PERFORMERS:")
display(top_5)
 
bottom_5 = gold_perf.orderBy(F.asc("Return_1M_Percent")).limit(5).select(
    "Ticker", "Return_1M_Percent", "Performance_Tier"
)
 
print("\n⚠️ BOTTOM 5 PERFORMERS:")
display(bottom_5)